# Circos Test

In [2]:
import altair as alt
from harpy.report.utilities import palette, trunc_digits
from harpy.report.theme import sv_colors
from IPython.display import display
import pandas as pd
import altair as alt
import pandas as pd
import numpy as np

In [10]:
# Sample data: structural variants (inversions) with start and end positions
inversions = pd.DataFrame({
    'chromosome': ['chr1', 'chr1', 'chr2', 'chr2', 'chr3', 'chr1', 'chr3'],
    'start': [1000000, 5000000, 2000000, 8000000, 3000000, 7000000, 6000000],
    'end': [2000000, 6500000, 3500000, 9500000, 5000000, 8500000, 7500000],
    'variant': ['INV','INV','INV','INV','INV','INV','INV']

})

# Sample data: deletions
deletions = pd.DataFrame({
    'chromosome': ['chr1', 'chr2', 'chr2', 'chr3', 'chr3', 'chr1', 'chr2'],
    'start': [4000, 1000000, 6000000, 1000000, 6000000, 8000000, 10000000],
    'end': [6000, 1800000, 7000000, 2000000, 7000000, 9000000, 11000000],
    'variant': ['DEL','DEL','DEL','DEL','DEL','DEL','DEL'],
})

# Define chromosome lengths (example values - adjust to your genome)
chr_lengths = {
    'chr1': 10000000,
    'chr2': 12000000,
    'chr3': 8000000
}

sv = pd.concat([inversions, deletions])

In [39]:
# Chromosome lengths
chr_lengths = {
    'chr1': 10000000,
    'chr2': 12000000,
    'chr3': 8000000
}

# Create dataframe
chrom_df = pd.DataFrame(list(chr_lengths.items()), columns=['chromosome', 'length'])

# Calculate cumulative positions (where each chromosome starts)
chrom_df['cum_start'] = np.insert(np.cumsum(chrom_df['length'].values)[:-1], 0, 0)
chrom_df['cum_end'] = chrom_df['cum_start'] + chrom_df['length']

# Total genome length
total_length = chrom_df['length'].sum()

# Convert to radians, leaving a 5% gap for visual separation
scale_factor = 1
chrom_df['theta_start'] = (chrom_df['cum_start'] / total_length) * 2 * np.pi * scale_factor
chrom_df['theta_end'] = (chrom_df['cum_end'] / total_length) * 2 * np.pi * scale_factor
#chrom_df['theta_mid'] = (chrom_df['theta_start'] + chrom_df['theta_end']) / 2

print(chrom_df)

# Create the circular chromosome ideogram
base_chart = alt.Chart(chrom_df).mark_arc(
    outerRadius=240,
    innerRadius=215,
    stroke='grey',
    strokeWidth=2,
    padAngle=0.02
).encode(
    theta=alt.Theta('theta_start:Q')
        .scale(alt.Scale(domain=[0, 2*np.pi])),
    theta2='theta_end:Q'
)

# Add chromosome labels
labels = (
    alt.Chart(chrom_df)
    .transform_calculate(theta_mid='(datum.theta_start + datum.theta_end) / 2')
    .mark_text(radius=270, size=14)
    .encode(
        theta='theta_mid:Q',
        text='chromosome:N'
    )
)

# Combine
final_chart = (
    base_chart + labels
    .properties(
        width=600,
        height=600,
        title='Circos Plot'
    )
)
final_chart

  chromosome    length  cum_start   cum_end  theta_start  theta_end
0       chr1  10000000          0  10000000     0.000000   2.094395
1       chr2  12000000   10000000  22000000     2.094395   4.607669
2       chr3   8000000   22000000  30000000     4.607669   6.283185


alt.LayerChart(...)

In [21]:
# Process inversions: merge with chromosome cumulative positions
inv_df = inversions.merge(
    chrom_df[['chromosome', 'cum_start']], 
    on='chromosome'
)

# Calculate absolute genomic positions
inv_df['abs_start'] = inv_df['cum_start'] + inv_df['start']
inv_df['abs_end'] = inv_df['cum_start'] + inv_df['end']

# Convert to radians
inv_df['theta_start'] = (inv_df['abs_start'] / total_length) * 2 * np.pi * scale_factor
inv_df['theta_end'] = (inv_df['abs_end'] / total_length) * 2 * np.pi * scale_factor

variants = alt.Chart(inv_df).mark_arc(
    outerRadius=215,
    innerRadius=205,
    stroke='white',
    strokeWidth=1
).encode(
    theta=alt.Theta('theta_start:Q', scale=alt.Scale(domain=[0, 2*np.pi])),
    theta2=alt.Theta2('theta_end:Q'),
    color=alt.Color('chromosome:N', legend=None),
    tooltip=['id:N', 'chromosome:N', 'start:Q', 'end:Q']
)

In [22]:

final_chart + variants

alt.LayerChart(...)

In [123]:

# Merge with chromosome info to get offsets
data = inversions.merge(_chrd[['chromosome', 'cumulative_start']], on='chromosome', suffixes=('', '_chr'))
data['var_start'] = data['cumulative_start'] + data['start']
data['var_end'] = data['cumulative_start'] + data['end']
data['theta'] = 1.96 * np.pi * data['var_start'] / total_length
data['theta2'] = 1.96 * np.pi * data['var_end']  / total_length
#data = data.drop('start_chr', axis=1)
data


,chromosome,start,end,id,variant,cumulative_start,var_start,var_end,theta,theta2
0,chr1,1000000,2000000,INV1,INV,0,1000000,2000000,0.205251,0.410501
1,chr1,5000000,6500000,INV2,INV,0,5000000,6500000,1.026254,1.334130
2,chr2,2000000,3500000,INV3,INV,10000000,12000000,13500000,2.463009,2.770885
3,chr2,8000000,9500000,INV4,INV,10000000,18000000,19500000,3.694513,4.002389
4,chr3,3000000,5000000,INV5,INV,22000000,25000000,27000000,5.131268,5.541769
5,chr1,7000000,8500000,INV6,INV,0,7000000,8500000,1.436755,1.744631
6,chr3,6000000,7500000,INV7,INV,22000000,28000000,29500000,5.747020,6.054896


In [103]:
def circos_grid(chr_lens: dict[str,int], rings: int):
    # Create dataframe
    chrom_df = pd.DataFrame(list(chr_lengths.items()), columns=['chromosome', 'length'])

    # Calculate cumulative positions (where each chromosome starts)
    chrom_df['cum_start'] = np.insert(np.cumsum(chrom_df['length'].values)[:-1], 0, 0)
    chrom_df['cum_end'] = chrom_df['cum_start'] + chrom_df['length']

    # Total genome length
    total_length = chrom_df['length'].sum()

    # Convert to radians, 
    #scale_factor = 0.95 # leaving a 5% gap for visual separation
    chrom_df['theta_start'] = (chrom_df['cum_start'] / total_length) * 2 * np.pi
    chrom_df['theta_end'] = (chrom_df['cum_end'] / total_length) * 2 * np.pi

    gap = 4
    radius_width = 35
    outer_radius = 240
    inner_radius = outer_radius - radius_width
    # Create the circular chromosome ideogram
    base_chart = (
        alt.Chart(chrom_df, width = 650, height = 650)
        .mark_arc(
            outerRadius= outer_radius,
            innerRadius= inner_radius,
            stroke='#4c4c4c',
            color='#eeeeee',
            filled=True,
            fillOpacity=1,
            strokeJoin="round",
            strokeWidth=1,
            padAngle=0.02
        )
        .encode(
            theta=alt.Theta('theta_start:Q')
                .scale(alt.Scale(domain=[0, 2*np.pi])),
            theta2='theta_end:Q'
        )
    )
    for i in range(rings-1):
        outer_radius = inner_radius - gap
        inner_radius = outer_radius - (radius_width * 0.9)
        base_chart += (
        alt.Chart(chrom_df)
        .mark_arc(
            outerRadius=outer_radius,
            innerRadius=inner_radius,
            stroke='#4c4c4c',
            color = '#eeeeee',
            strokeJoin="round",
            filled=True,
            fillOpacity=1,
            strokeWidth=1,
            padAngle=0.02
        )
        .encode(
            theta=alt.Theta('theta_start:Q')
                .scale(alt.Scale(domain=[0, 2*np.pi])),
            theta2='theta_end:Q'
        )
    )

    # Add chromosome labels
    labels = (
        alt.Chart(chrom_df)
        .transform_calculate(theta_mid='(datum.theta_start + datum.theta_end) / 2')
        .mark_text(radius=270, size=14, fill = "black")
        .encode(
            theta='theta_mid:Q',
            text='chromosome:N'
        )
    )

    return (
        base_chart + labels
        .properties(title='Structural Variants')
    ).configure(background = "white")


In [104]:
circos_grid(chr_lengths, rings = 4)

alt.LayerChart(...)

In [52]:
def radian_positions(chr_lens: dict[str,int]) -> pd.DataFrame:
    '''
    Convert the dict of chromosome names and lengths to a DataFrame
    that's in polar (radian) coordinates.
    '''
    # Create dataframe
    chrom_df = pd.DataFrame(list(chr_lens.items()), columns=['chromosome', 'length'])

    # Calculate cumulative positions (where each chromosome starts)
    chrom_df['cum_start'] = np.insert(np.cumsum(chrom_df['length'].values)[:-1], 0, 0)
    chrom_df['cum_end'] = chrom_df['cum_start'] + chrom_df['length']

    # Total genome length
    total_length = chrom_df['length'].sum()

    # Convert to radians, 
    #scale_factor = 0.95 # leaving a 5% gap for visual separation
    chrom_df['theta_start'] = (chrom_df['cum_start'] / total_length) * 2 * np.pi
    chrom_df['theta_end'] = (chrom_df['cum_end'] / total_length) * 2 * np.pi
    return chrom_df

def circos_grid(chr_df: pd.DataFrame, rings: int):
    '''
    Given the polar-converted chromosome grid `chr_df` (created by `radian_positions`),
    return the circos-style base plot with chromosome names and grides.
    '''
    gap = 4
    radius_width = 35
    outer_radius = 240
    inner_radius = outer_radius - radius_width
    # Create the circular chromosome ideogram
    base_chart = (
        alt.Chart(chr_df, width = 650, height = 650)
        .mark_arc(
            outerRadius= outer_radius,
            innerRadius= inner_radius,
            stroke='#bebebe',
            color='#f4f4f4',
            filled=True,
            fillOpacity=1,
            strokeJoin="round",
            strokeWidth=1,
            padAngle=0.01
        )
        .encode(
            theta=alt.Theta('theta_start:Q')
                .scale(alt.Scale(domain=[0, 2*np.pi])),
            theta2='theta_end:Q'
        )
    )
    for _ in range(rings-1):
        outer_radius = inner_radius - gap
        inner_radius = outer_radius - (radius_width * 0.9)
        base_chart += (
        alt.Chart(chr_df)
        .mark_arc(
            outerRadius=outer_radius,
            innerRadius=inner_radius,
            stroke='#bebebe',
            color = '#f4f4f4',
            strokeJoin="round",
            filled=True,
            fillOpacity=1,
            strokeWidth=1,
            padAngle=0.01
        )
        .encode(
            theta=alt.Theta('theta_start:Q')
                .scale(alt.Scale(domain=[0, 2*np.pi])),
            theta2='theta_end:Q'
        )
    )

    # Add chromosome labels
    labels = (
        alt.Chart(chr_df)
        .transform_calculate(theta_mid='(datum.theta_start + datum.theta_end) / 2')
        .mark_text(radius=270, size=14, fill = "black", padAngle=0.02)
        .encode(
            theta='theta_mid:Q',
            text='chromosome:N'
        )
    )

    return (
        base_chart + labels
        .properties(title='Structural Variants')
    )

def variant_bands(var_df: pd.DataFrame, baseplot):
    gap = 4
    radius_width = 35
    outer_radius = 240
    inner_radius = outer_radius - radius_width
    var_groups = var_df.groupby('variant', sort = False)
    for variant, gdf in var_groups:
        _color = sv_colors(variant)
        baseplot += (
            alt.Chart(gdf)
            .mark_arc(
                outerRadius=outer_radius - 3,
                innerRadius=inner_radius + 3,
                stroke=_color,
                strokeWidth=2,
                strokeJoin="round",
                color = _color,
                thetaOffset=0.01,
                theta2Offset=0.01
            )
        .encode(
            theta=alt.Theta('theta_start:Q', scale=alt.Scale(domain=[0, 2*np.pi])),
            theta2='theta_end:Q',
            tooltip=['variant:N', 'chromosome:N', 'start:Q', 'end:Q']
        )
            )
        outer_radius = inner_radius - gap
        inner_radius = outer_radius - (radius_width * 0.9)

    return baseplot

In [54]:
chrom_df = radian_positions(chr_lengths)
total_length = chrom_df['length'].sum()

# Process inversions: merge with chromosome cumulative positions
var_df = sv.merge(chrom_df[['chromosome', 'cum_start']], on='chromosome')

# Calculate absolute genomic positions
var_df['abs_start'] = var_df['cum_start'] + var_df['start']
var_df['abs_end'] = var_df['cum_start'] + var_df['end']

# Convert to radians
var_df['theta_start'] = (var_df['abs_start'] / total_length) * 2 * np.pi
var_df['theta_end'] = (var_df['abs_end'] / total_length) * 2 * np.pi

# create plot base/grid
plot = circos_grid(chrom_df, var_df['variant'].nunique())
(plot + variant_bands(var_df, plot)).configure(background = "white")

alt.LayerChart(...)